# 04 — Watchdog: Keep Silver Streaming Alive

Este notebook es invocado por un **Data Pipeline** cada 5 minutos.
Verifica si `01-silver-streaming` tiene un job activo en Fabric.
Si no está corriendo, lo lanza vía la Fabric REST API.

**No requiere lakehouse default.**  
**No modificar las celdas — solo ajustar parámetros si cambia el nombre del notebook.**

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
STREAMING_NOTEBOOK_NAME = "01-silver-streaming"  # display name in Fabric workspace

In [ ]:
# ── Resolve workspace ID and auth token ───────────────────────────────────────
# spark.conf 'trident.workspace.id' is injected by Fabric into every Spark session
import requests
import notebookutils

workspace_id = spark.conf.get("trident.workspace.id")
token        = notebookutils.credentials.getToken("https://api.fabric.microsoft.com/")
headers      = {
    "Authorization": f"Bearer {token}",
    "Content-Type":  "application/json",
}
base_url = "https://api.fabric.microsoft.com/v1"

print(f"Workspace ID : {workspace_id}")

In [ ]:
# ── Find streaming notebook item ID by display name ───────────────────────────
resp = requests.get(f"{base_url}/workspaces/{workspace_id}/items", headers=headers)
resp.raise_for_status()

items = resp.json().get("value", [])
notebook_item = next(
    (i for i in items
     if i["displayName"] == STREAMING_NOTEBOOK_NAME and i["type"] == "Notebook"),
    None,
)

if notebook_item is None:
    raise RuntimeError(f"Notebook '{STREAMING_NOTEBOOK_NAME}' not found in workspace {workspace_id}")

notebook_id = notebook_item["id"]
print(f"Notebook     : {STREAMING_NOTEBOOK_NAME} ({notebook_id})")

In [ ]:
# ── Check for active job instances ────────────────────────────────────────────
jobs_url = (
    f"{base_url}/workspaces/{workspace_id}/items/{notebook_id}"
    f"/jobs/instances?jobType=RunNotebook"
)
resp = requests.get(jobs_url, headers=headers)
resp.raise_for_status()

jobs    = resp.json().get("value", [])
running = [j for j in jobs if j.get("status") in ("InProgress", "NotStarted")]

print(f"Total jobs found : {len(jobs)}")
print(f"Active jobs      : {len(running)}")

In [ ]:
# ── Launch if not running ─────────────────────────────────────────────────────
if running:
    job = running[0]
    print(f"Streaming notebook already running — job id: {job['id']}, status: {job['status']}")
    print("Nothing to do.")
else:
    print("Streaming notebook is NOT running. Launching...")
    trigger_url = (
        f"{base_url}/workspaces/{workspace_id}/items/{notebook_id}"
        f"/jobs/instances?jobType=RunNotebook"
    )
    resp = requests.post(trigger_url, headers=headers, json={})

    if resp.status_code in (200, 202):
        location = resp.headers.get("Location", "(no location header)")
        print(f"Notebook launched successfully. HTTP {resp.status_code}")
        print(f"Job location : {location}")
    else:
        raise RuntimeError(
            f"Failed to launch notebook. HTTP {resp.status_code}: {resp.text}"
        )